# Assignment 4 Reproduction Study

**Student:** Adilbekov Daniyal  
**Group:** se-2414  
**Project:** Sequential Knowledge Editing with ROME and MEMIT on GPT-2 XL

This notebook evaluates the reproducibility of the experiments released under the `assignment-4` tag.
The protocol comprises 100 sequential CounterFact edits, two editing methods and three fixed edit orders.
Each run starts from the original GPT-2 XL weights. Evaluation takes place after 10, 25, 50 and 100 edits.

The outputs include run logs, environment records, checkpoint comparisons, summary statistics, figures
and the reproduction record for Appendix C. The experiment source code remains unchanged.

**Repository:** https://github.com/Janik07/ke-rome-memit


## 1 Runtime and Session

Google Colab with an NVIDIA Tesla T4. Each session has a separate results directory on Google Drive.
Environment metadata and session timestamps are recorded in the Asia/Almaty time zone.


In [ ]:
import os, sys, json, subprocess, platform, hashlib, shutil, csv, shlex
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

PERSON = 'Adilbekov Daniyal'
SESSION_NAME = ''
TAG = 'assignment-4'
EXPECTED_COMMIT = '03e5a68fd40d84eb4d121658b52bfef0d95ddb8c'
EE_COMMIT = '4c109870955a4522ac3d7cf10ad00f34de8e4f0d'
TZ = ZoneInfo('Asia/Almaty')
def now():
    return datetime.now(TZ).isoformat(timespec='seconds')

gpu_text = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    text=True).strip()
print('GPU:', gpu_text)
assert 'T4' in gpu_text, 'An NVIDIA Tesla T4 runtime is required.'
from google.colab import drive
drive.mount('/content/drive')
base = Path('/content/drive/MyDrive/KE_Reproduction_Adilbekov')
local_session = Path('/content/ke_reproduction_session.txt')
if not SESSION_NAME:
    SESSION_NAME = (local_session.read_text().strip() if local_session.exists()
                    else datetime.now(TZ).strftime('%Y%m%d_%H%M%S'))
assert Path(SESSION_NAME).name == SESSION_NAME and SESSION_NAME not in {'.', '..'}
local_session.write_text(SESSION_NAME)
SESSION = base / SESSION_NAME
OUT = SESSION / 'results'
LOGS = SESSION / 'logs'
META = SESSION / 'metadata'
for p in (OUT, LOGS, META):
    p.mkdir(parents=True, exist_ok=True)
REPO = Path('/content/ke-rome-memit')
EE = Path('/content/EasyEdit')
STATS = EE / 'data/stats'
CF = Path('/content/counterfact.json')
def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2))
identity_path = META / 'identity.json'
if identity_path.exists():
    identity = json.loads(identity_path.read_text())
    assert identity['commit'] == EXPECTED_COMMIT and identity['person'] == PERSON
else:
    identity = {'person': PERSON, 'started_at': now(), 'tag': TAG, 'commit': EXPECTED_COMMIT}
    write_json(identity_path, identity)
write_json(META / ('session_entry_' + datetime.now(TZ).strftime('%Y%m%d_%H%M%S') + '.json'),
           {'time': now(), 'python': sys.version, 'os': platform.platform(), 'gpu': gpu_text})
print('SESSION_NAME =', SESSION_NAME)
print('Session directory:', SESSION)


## 2 Source Versions and Command Logging

Project release: `assignment-4`, commit `03e5a68fd40d84eb4d121658b52bfef0d95ddb8c`.
EasyEdit commit: `4c109870955a4522ac3d7cf10ad00f34de8e4f0d`.
Commands, working directories, console output and exit codes are retained for traceability.


In [ ]:
ENV = os.environ.copy()
ENV.update({'EE_DIR': str(EE), 'KE_OUT': str(OUT), 'KE_STATS': str(STATS),
            'KE_CF': str(CF), 'PYTHONUNBUFFERED': '1'})
def run_cmd(args, cwd=None, label='command', progress_only=False):
    args = [str(x) for x in args]
    started = now()
    log = LOGS / (label + '_' + datetime.now(TZ).strftime('%Y%m%d_%H%M%S_%f') + '.log')
    record = {'started_at': started, 'args': args, 'cwd': str(cwd or '/content'),
              'log_file': str(log.relative_to(SESSION))}
    print('$', shlex.join(args), flush=True)
    tail = []
    rc = None
    proc = None
    try:
        with log.open('w') as f:
            proc = subprocess.Popen(args, cwd=str(cwd or '/content'), env=ENV,
                                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                    text=True, bufsize=1)
            for line in proc.stdout:
                f.write(line)
                f.flush()
                tail = (tail + [line])[-35:]
                if not progress_only or line.startswith(('[baseline]', '[rome_', '[memit_')):
                    print(line, end='', flush=True)
            rc = proc.wait()
    finally:
        if proc is not None and proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
        record.update({'finished_at': now(), 'returncode': rc})
        with (META / 'commands.jsonl').open('a') as f:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
    if rc != 0:
        if progress_only:
            print(''.join(tail))
        raise RuntimeError(f'Command failed with exit code {rc}. Log: {log}')
    return log

def clone_checked(url, dest, ref):
    if not dest.exists():
        run_cmd(['git', 'clone', url, dest], label='clone_' + dest.name)
    assert (dest / '.git').exists(), f'{dest} exists but is not a Git repository.'
    origin = subprocess.check_output(['git', '-C', str(dest), 'remote', 'get-url', 'origin'], text=True).strip()
    assert origin.removesuffix('.git') == url.removesuffix('.git'), 'Unexpected repository origin: ' + origin
    clean = subprocess.check_output(['git', '-C', str(dest), 'status', '--porcelain', '--untracked-files=no'], text=True)
    assert not clean.strip(), f'Modified tracked files detected in {dest}.'
    run_cmd(['git', '-C', dest, 'checkout', '--detach', ref], label='checkout_' + dest.name)

clone_checked('https://github.com/Janik07/ke-rome-memit.git', REPO, TAG)
actual_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == EXPECTED_COMMIT, 'The release tag does not match the expected commit.'
clone_checked('https://github.com/zjunlp/EasyEdit.git', EE, EE_COMMIT)
assert subprocess.check_output(['git', '-C', str(EE), 'rev-parse', 'HEAD'], text=True).strip() == EE_COMMIT
print('Project commit:', actual_commit)
print('EasyEdit commit:', EE_COMMIT)


## 3 Dependencies and Environment

Dependencies are installed from the project requirements file. Python, PyTorch, Transformers,
CUDA and GPU information are recorded after installation. Model processes are isolated from the notebook kernel.


In [ ]:
run_cmd([sys.executable, '-m', 'pip', 'install', '-r', REPO / 'requirements.txt'], label='pip_install')
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(META / 'pip_freeze.txt').write_text(freeze)
probe = "import torch, transformers, sys, platform, json; print(json.dumps({'python':sys.version,'os':platform.platform(),'torch':torch.__version__,'transformers':transformers.__version__,'cuda_build':torch.version.cuda,'cuda_available':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"
environment = json.loads(subprocess.check_output([sys.executable, '-c', probe], text=True, env=ENV))
assert environment['cuda_available'], 'CUDA is unavailable in PyTorch.'
assert 'T4' in environment['gpu'], 'An NVIDIA Tesla T4 runtime is required.'
environment.update({'recorded_at': now(), 'nvidia_smi': gpu_text})
env_path = META / ('environment_' + datetime.now(TZ).strftime('%Y%m%d_%H%M%S') + '.json')
write_json(env_path, environment)
print(json.dumps(environment, ensure_ascii=False, indent=2))
run_cmd([sys.executable, '-c', 'from easyeditor import ROMEHyperParams, MEMITHyperParams; print("EasyEdit imports OK")'],
        cwd=EE, label='easyedit_import_check')


## 4 Data and Covariance Statistics

CounterFact and the precomputed Wikipedia covariance statistics for GPT-2 XL layers 13–17
are retrieved from the ROME authors' data repository. The CounterFact file hash is recorded.


In [ ]:
def download(url, dest):
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size:
        print('Cached file:', dest.name)
        return
    temporary = dest.with_suffix(dest.suffix + '.part')
    run_cmd(['wget', '--no-verbose', '--tries=3', '--timeout=60', url, '-O', temporary], label='download_' + dest.stem)
    assert temporary.stat().st_size > 0
    temporary.replace(dest)

download('https://rome.baulab.info/data/dsets/counterfact.json', CF)
cf = json.loads(CF.read_text())
assert isinstance(cf, list) and len(cf) > 100, 'Unexpected CounterFact data format.'
del cf
for layer in range(13, 18):
    filename = f'transformer.h.{layer}.mlp.c_proj_float32_mom2_100000.npz'
    download('https://rome.baulab.info/data/stats/gpt2-xl/wikipedia_stats/' + filename,
             STATS / 'gpt2-xl/wikipedia_stats' / filename)
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
write_json(META / 'data_source.json', {'counterfact_sha256': sha256(CF), 'bytes': CF.stat().st_size})
print('Data download completed.')


## 5 Sample and Edit Order Verification

The 100-record sample and all three edit orders are regenerated and compared with the release sample.
Completed runs are validated against their saved checksums. Incomplete attempts are archived separately.


In [ ]:
run_cmd([sys.executable, 'run_experiment.py', 'prepare'], cwd=REPO / 'src', label='prepare')
sample = json.loads((OUT / 'sample.json').read_text())
reference_sample = json.loads((REPO / 'results/sample.json').read_text())
assert sample == reference_sample, 'The generated sample or edit orders differ from the reference.'
print('All 100 records and three edit orders match the reference.')

CPS = ['10', '25', '50', '100']
def run_complete(data, method, order):
    return (data.get('method') == method and data.get('order_id') == order
            and len(data.get('edits', [])) == 100
            and all(c in data.get('checkpoints_eval', {}) for c in CPS)
            and data.get('total_seconds', 0) > 0
            and data.get('order') == sample['orders'][order])

def run_one(method, order):
    name = f'{method}_order{order}'
    output = OUT / (name + '.json')
    marker = META / (name + '_completed.json')
    if output.exists() and marker.exists():
        saved = json.loads(marker.read_text())
        if (saved.get('commit') == EXPECTED_COMMIT and saved.get('sha256') == sha256(output)
                and run_complete(json.loads(output.read_text()), method, order)):
            print(name, 'already completed and validated in this session.')
            return
    if output.exists():
        archive = SESSION / 'previous_attempts'
        archive.mkdir(exist_ok=True)
        shutil.copy2(output, archive / (name + '_' + datetime.now(TZ).strftime('%Y%m%d_%H%M%S') + '.json'))
    started = now()
    run_cmd([sys.executable, 'run_experiment.py', 'run', method, str(order)],
            cwd=REPO / 'src', label=name, progress_only=True)
    result = json.loads(output.read_text())
    assert run_complete(result, method, order), 'The experiment output is incomplete.'
    write_json(marker, {'started_at': started, 'finished_at': now(), 'commit': EXPECTED_COMMIT,
                        'sha256': sha256(output), 'environment_file': env_path.name})
    print('Run output saved:', output)


## Run 1 MEMIT Order 0


In [ ]:
run_one('memit', 0)


## Run 2 ROME Order 0


In [ ]:
run_one('rome', 0)


## Run 3 MEMIT Order 1


In [ ]:
run_one('memit', 1)


## Run 4 ROME Order 1


In [ ]:
run_one('rome', 1)


## Run 5 MEMIT Order 2


In [ ]:
run_one('memit', 2)


## Run 6 ROME Order 2


In [ ]:
run_one('rome', 2)


## 6 Statistical Analysis and Figures

The original analysis scripts produce the summary statistics, error analysis and figures from all six completed runs.


In [ ]:
for order in range(3):
    for method in ('rome', 'memit'):
        p = OUT / f'{method}_order{order}.json'
        assert p.exists() and run_complete(json.loads(p.read_text()), method, order), f'Incomplete run: {p.name}'
run_cmd([sys.executable, 'analyze.py', OUT], cwd=REPO / 'src', label='analyze')
run_cmd([sys.executable, 'error_analysis.py', OUT], cwd=REPO / 'src', label='error_analysis')
run_cmd([sys.executable, 'plots.py', OUT, SESSION / 'figures'], cwd=REPO / 'src', label='plots')


## 7 Reproduction Comparison

The comparison covers edit order, per-edit success, immediate success rate, baseline values
and checkpoint metrics. The regenerated summary is also compared, including retention statistics.
Execution times are recorded separately and excluded from metric agreement checks.

The strict comparison reports **Yes** when all six runs and the summary match, **Partly** when at least
one run matches but coverage or agreement is incomplete, and **No** when no completed run matches in full.
Numeric agreement uses an absolute tolerance of `1e-9` on the stored values.


In [ ]:
def same(a, b):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        import math
        return math.isfinite(a) and math.isfinite(b) and abs(a - b) <= 1e-9
    return a == b

def compare_run(ref, actual, name):
    rows = []
    def add(checkpoint, metric, expected, observed):
        rows.append({'run': name, 'checkpoint': checkpoint, 'metric': metric,
                     'expected': expected, 'observed': observed, 'match': same(expected, observed)})
    add('all', 'order_matches', True, ref.get('order') == actual.get('order'))
    re = [e['success'] for e in ref['edits']]
    ae = [e['success'] for e in actual['edits']]
    add('all', 'success_flags_match', True, re == ae)
    add('all', 'immediate_success_rate', sum(re) / len(re), sum(ae) / len(ae))
    for metric in ('rewrite_rate', 'para_rate', 'neigh_rate', 'n'):
        add('baseline', metric, ref['baseline']['summary'][metric], actual['baseline']['summary'][metric])
    for cp in CPS:
        for metric in ('rewrite_rate', 'para_rate', 'neigh_rate', 'n'):
            add(cp, metric, ref['checkpoints_eval'][cp]['summary'][metric],
                actual['checkpoints_eval'][cp]['summary'][metric])
    return rows

comparison = []
complete_names = []
matching_runs = []
for order in range(3):
    for method in ('rome', 'memit'):
        name = f'{method}_order{order}'
        p = OUT / (name + '.json')
        if not p.exists():
            continue
        actual = json.loads(p.read_text())
        if not run_complete(actual, method, order):
            continue
        marker = META / (name + '_completed.json')
        if not marker.exists() or json.loads(marker.read_text()).get('sha256') != sha256(p):
            continue
        complete_names.append(name)
        ref = json.loads((REPO / 'results' / p.name).read_text())
        rows = compare_run(ref, actual, name)
        comparison.extend(rows)
        if all(r['match'] for r in rows):
            matching_runs.append(name)

summary_differences = []
IGNORE_KEYS = {'mean_seconds_per_edit', 'total_minutes'}
def compare_tree(expected, observed, path='summary'):
    if isinstance(expected, dict) and isinstance(observed, dict):
        for key in sorted(set(expected) | set(observed)):
            if key not in IGNORE_KEYS:
                compare_tree(expected.get(key), observed.get(key), path + '.' + key)
    elif isinstance(expected, list) and isinstance(observed, list):
        if len(expected) != len(observed):
            summary_differences.append({'path':path, 'expected':expected, 'observed':observed})
        else:
            for i, (a, b) in enumerate(zip(expected, observed)):
                compare_tree(a, b, path + f'[{i}]')
    elif not same(expected, observed):
        summary_differences.append({'path':path, 'expected':expected, 'observed':observed})

summary_exists = (OUT / 'summary.json').exists()
if summary_exists and len(complete_names) == 6:
    compare_tree(json.loads((REPO / 'results/summary.json').read_text()),
                 json.loads((OUT / 'summary.json').read_text()))
all_checks_match = bool(comparison) and all(r['match'] for r in comparison)
status = ('Yes' if len(complete_names) == 6 and all_checks_match and summary_exists and not summary_differences
          else 'Partly' if matching_runs else 'No')
comparison_path = SESSION / 'comparison.csv'
with comparison_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['run','checkpoint','metric','expected','observed','match'])
    writer.writeheader()
    writer.writerows(comparison)
write_json(SESSION / 'summary_differences.json', summary_differences)
facts = {'person': PERSON, 'recorded_at': now(), 'tag': TAG, 'commit': actual_commit,
         'easyedit_commit': EE_COMMIT, 'completed_runs': complete_names,
         'fully_matching_runs': matching_runs, 'suggested_status': status,
         'metric_mismatches': sum(not r['match'] for r in comparison),
         'summary_comparison_done': summary_exists and len(complete_names) == 6,
         'summary_differences': summary_differences,
         'environment': environment,
         'notes': ['A new sample and all edit orders were generated and compared with results/sample.json.',
                   'Project cloning and explicit checkout of assignment-4 were added to the README procedure.',
                   'README <this-repo> was replaced with /content/ke-rome-memit.',
                   'KE_OUT points to this Google Drive session to preserve the new logs.',
                   'Runtime differences are excluded from exact quality-metric comparison.',
                   'The original analyze.py rounds very small p-values to zero; source code was not modified.',
                   'Console logs and command exit codes are retained for each executed command.']}
write_json(SESSION / 'appendix_c_facts.json', facts)
commands = [json.loads(line) for line in (META / 'commands.jsonl').read_text().splitlines() if line.strip()]
commands_text = '\n'.join(f"{r['started_at']} | cwd={r['cwd']} | {shlex.join(r['args'])} | exit={r['returncode']}" for r in commands)
(SESSION / 'commands_executed.txt').write_text(commands_text)
record = (f"Person who reproduced the results: {PERSON}\n"
          f"Test started: {identity['started_at']}\nRecord created: {now()}\n"
          f"Machine and operating system: Google Colab; {environment['gpu']}; {environment['os']}\n"
          f"Python: {environment['python']}\nPyTorch: {environment['torch']}\nTransformers: {environment['transformers']}\n"
          f"Commit / release tag used: {TAG} / {actual_commit}\n"
          f"Commands executed: commands_executed.txt, including recorded failures.\n"
          f"Completed runs: {', '.join(complete_names) or 'None'}\n"
          f"Did the main results table reproduce? Suggested status: {status}\n"
          f"Differences observed: {facts['metric_mismatches']} failed run-level checks; "
          f"{len(summary_differences)} summary differences. Detailed records: comparison.csv and summary_differences.json.\n"
          f"The comparison records numerical agreement; causes of any discrepancies are not inferred automatically.\n"
          f"Problems in README / procedure: " + ' '.join(facts['notes']) + '\n')
(SESSION / 'Appendix_C_record.txt').write_text(record)
print(record)
try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(comparison))
except ImportError:
    for row in comparison:
        print(row)
print('Reproduction comparison and Appendix C record saved.')


## 8 Results Archive

The archive contains the current session's outputs, command logs, environment metadata and reproduction record.
Model weights and the reference run files are excluded.


In [ ]:
from google.colab import files
archive = shutil.make_archive('/content/KE_Reproduction_Adilbekov_' + SESSION_NAME, 'zip', root_dir=SESSION)
shutil.copy2(archive, SESSION.parent / Path(archive).name)
print('Archive:', archive)
print('Archive copy saved to Google Drive.')
files.download(archive)
